<img src="https://www.unad.edu.co/images/footer/logo-unad-acreditacion-min.png" width="780" height="140" align="right"/>

<p style="text-align: center;"> Curso: ENSEMBLE METHODS AND KERNELS</p>

<p style="text-align: center;"> Código Curso: 203008076 </p>

<p style="text-align: center;"> Grupo: 1 </p>

<p style="text-align: center;"> Phase 3 -Development of the Practical Component of the
Course Ensemble Methods and Kernels</p>

<p style="text-align: center;">  Presentado por: Wilmer Ricardo Urda</p>

<p style="text-align: center;"> Código: 1017194627</p>

<p style="text-align: center;">  Tutor: Ing. Jorge Luis Quintero Lopez </p>

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>


#Exercise 1: Bagging Method

##BAGGING PARA REGRESIÓN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================
# CARGAR DATASET (meta - ID 566)
# =========================
data = fetch_openml(data_id=566, as_frame=True)
df = data.frame.copy()

target_col = data.target_names[0]
X = df.drop(columns=[target_col])
y = pd.to_numeric(df[target_col], errors="coerce")

mask = y.notna()
X, y = X[mask], y[mask]

# =========================
# PREPROCESAMIENTO CON PIPELINE
# =========================
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# MODELO BAGGING
# =========================
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("bagging", BaggingRegressor(
        estimator=DecisionTreeRegressor(),
        n_estimators=50,
        random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# =========================
# MÉTRICAS
# =========================
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
metrics_df = pd.DataFrame({"Metric": ["R²", "RMSE"], "Value": [round(r2, 4), round(rmse, 4)]})
print(metrics_df.to_string(index=False))

# =========================
# LEARNING CURVE
# =========================
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y,
    cv=5,
    scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Train R²")
plt.plot(train_sizes, test_scores.mean(axis=1), marker="o", label="Test R²")
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes, test_scores.mean(axis=1) - test_scores.std(axis=1),
                 test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.15)
plt.title("Learning Curve - Bagging Regressor (meta dataset)")
plt.xlabel("Training Size")
plt.ylabel("R²")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


##BAGGING PARA CLASIFICACIÓN

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================
# CARGAR DATASET (kr-vs-kp - ID 3)
# =========================
data_clf = fetch_openml(data_id=3, as_frame=True)
df_clf = data_clf.frame.copy()

target_col_clf = data_clf.target_names[0]
X_clf = df_clf.drop(columns=[target_col_clf])
y_clf = LabelEncoder().fit_transform(df_clf[target_col_clf])

# =========================
# PREPROCESAMIENTO CON PIPELINE (evita data leakage)
# =========================
cat_cols_clf = X_clf.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols_clf = X_clf.select_dtypes(include=[np.number]).columns.tolist()

preprocessor_clf = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), num_cols_clf),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols_clf)
])

# =========================
# SPLIT (con stratify para balance de clases)
# =========================
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# =========================
# MODELO BAGGING
# =========================
model_clf = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("bagging", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=50,
        random_state=42
    ))
])

model_clf.fit(X_train_clf, y_train_clf)
y_pred_clf = model_clf.predict(X_test_clf)

# =========================
# MÉTRICAS
# =========================
metrics_clf = pd.DataFrame([{
    "Accuracy":  round(accuracy_score(y_test_clf, y_pred_clf), 4),
    "Precision": round(precision_score(y_test_clf, y_pred_clf, average="weighted"), 4),
    "Recall":    round(recall_score(y_test_clf, y_pred_clf, average="weighted"), 4),
    "F1-score":  round(f1_score(y_test_clf, y_pred_clf, average="weighted"), 4),
}])
print(metrics_clf.to_string(index=False))

# =========================
# LEARNING CURVE
# =========================
train_sizes_clf, train_scores_clf, test_scores_clf = learning_curve(
    model_clf, X_clf, y_clf,
    cv=5,
    scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes_clf, train_scores_clf.mean(axis=1), marker="o", label="Train Accuracy")
plt.plot(train_sizes_clf, test_scores_clf.mean(axis=1), marker="o", label="Test Accuracy")
plt.fill_between(train_sizes_clf,
                 train_scores_clf.mean(axis=1) - train_scores_clf.std(axis=1),
                 train_scores_clf.mean(axis=1) + train_scores_clf.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes_clf,
                 test_scores_clf.mean(axis=1) - test_scores_clf.std(axis=1),
                 test_scores_clf.mean(axis=1) + test_scores_clf.std(axis=1), alpha=0.15)
plt.title("Learning Curve - Bagging Classifier (kr-vs-kp dataset)")
plt.xlabel("Training Size")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


##Bagging – Regression

## Bagging - Regression

The Bagging Regressor was implemented using a full `Pipeline` that includes preprocessing (median imputation for numerical features and one-hot encoding for categorical features such as `DS_Name` and `Alg_Name`) followed by a `BaggingRegressor` with 50 estimators.

This approach ensures that feature encoding is fitted only on the training set, avoiding data leakage that would arise from transforming the full dataset before splitting.

The learning curve shows that:

- The training R2 score starts high and stabilizes as more data is used.
- The test R2 improves progressively, indicating better generalization with more training samples.
- The shaded confidence bands show model stability across cross-validation folds.
- The gap between training and test scores is reduced compared to a single decision tree, confirming that Bagging effectively mitigates overfitting through variance reduction.

Increasing `n_estimators` from 10 to 50 provides more stable bootstrap aggregation, reducing the randomness inherent in using few estimators.

##Bagging – Classification

## Bagging - Classification

The Bagging Classifier was implemented using a `Pipeline` that applies one-hot encoding to all categorical features of the `kr-vs-kp` dataset via `ColumnTransformer`, avoiding the data leakage that arises when applying `pd.get_dummies` before the train/test split.

The split was performed with `stratify=y` to ensure both training and test sets preserve the original class distribution.

The learning curve indicates that:

- Training accuracy is consistently high across all training sizes.
- Test accuracy improves and converges toward the training accuracy as more data is used, showing stable generalization.
- The narrow confidence bands confirm low variance in the model's behavior across folds.

This confirms that Bagging is effective in reducing overfitting and improving classification stability, particularly when combined with a proper preprocessing pipeline.